# Hodgkin–Huxley action potential

*biophysics · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/biophysics/hodgkin_huxley)

The 1952 model of the squid giant axon: a membrane capacitor discharged by sodium and potassium channels whose gates $m$, $h$, $n$ open and close with the voltage. Forty-one axons, each driven by a different current step, are integrated together in one $(4, B)$ tensor to find the spike threshold and the firing rate against current.

**Vector form**

$$C\,\frac{dV}{dt} = I - \bar g_{\mathrm{Na}}\,m^3 h\,(V - E_{\mathrm{Na}}) - \bar g_{\mathrm{K}}\,n^4\,(V - E_{\mathrm{K}}) - \bar g_L\,(V - E_L),\qquad \frac{dx}{dt} = \phi\,\big[\alpha_x(V)(1 - x) - \beta_x(V)\,x\big],\quad x \in \{m, h, n\}$$

**Index (tensor) form**

$$\Gamma_{kb} = \big(m_b^3 h_b,\ n_b^4,\ 1\big) \qquad C\,\dot V_b = I_b - \bar g_k\,\Gamma_{kb}\,(V_b - E_k) \qquad \dot x_{gb} = \phi\,\big[\alpha_g(V_b)(1 - x_{gb}) - \beta_g(V_b)\,x_{gb}\big]$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $I_b$ | `I_ext` | (B,) | stimulus current of axon $b$, a step switched on at 10 ms (µA/cm²) |
| $V_b,\ x_{gb}$ | `y[0], y[1:]` | (4, B) | membrane potential and the gates $g = (m, h, n)$ of every axon |
| $\alpha_g(V_b),\ \beta_g(V_b)$ | `rates(V)` | (3, B) | opening and closing rates of each gate (1/ms), the classic squid-axon fits |
| $\Gamma_{kb}$ | `open_fraction(gates)` | (3, B) | open fraction of channel $k$ = Na, K, leak |
| $\bar g_k,\ E_k$ | `g, E` | (3,) | maximal conductances (120, 36, 0.3 mS/cm²) and reversal potentials (50, −77, −54.4 mV) |
| $\bar g_k\,\Gamma_{kb}\,(V_b - E_k)$ | `einsum('k,kb,kb->b', g, Gamma, V - E[:, None])` | (B,) | total ionic current; repeated $k$ ⇒ summed over the channels |
| $\phi$ | `3 ** ((temp - 6.3) / 10)` | scalar | temperature factor ($Q_{10} = 3$): every gate speeds up 3× per 10 °C |
| $f_b$ | `firing_rate(spike_times(times, V))` | (B,) | steady firing rate from the interspike intervals (Hz) |

Hodgkin and Huxley (Nobel Prize 1963) measured the sodium and potassium currents of the squid giant axon with a voltage clamp and fitted them with three gates. The sodium current switches on fast through $m^3$ and off more slowly through the inactivation gate $h$; the potassium current opens late through $n^4$. A spike is the race between them: a small depolarization opens $m$, the inflowing $\mathrm{Na}^+$ depolarizes further (positive feedback) and $V$ shoots toward $E_{Na}$; then $h$ closes and $n$ opens, and $V$ falls back and even undershoots. With a sustained current step the model shows three regimes. Below about 2.3 µA/cm² the membrane only charges up passively. Between that and about 6.3 µA/cm² it fires a single spike and then settles, because inactivation and the potassium current hold it at a new rest. Above that it fires repetitively, and the rate does not start from zero but jumps to about 50 Hz: the squid axon is a type II (resonator) neuron whose resting state loses stability in a Hopf bifurcation, unlike type I neurons that can fire arbitrarily slowly. The f–I curve then rises gently, to about 68 Hz at 10 µA/cm². The rate functions have removable singularities, $0.1(V+40)/(1 - e^{-(V+40)/10})$ is finite at $V = -40$ mV, so the code evaluates them through $x/(1 - e^{-x})$ with a safe limit. The original experiments were done at 6.3 °C; warming the axon speeds up every gate by $\phi = 3^{(T - 6.3)/10}$, which makes the spikes narrower and the firing faster, until above about 30 °C the axon fails to fire at all. Simplifications: a space-clamped patch (no propagation along the axon) and deterministic channels.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/biophysics/hodgkin_huxley/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    g, E = tensors.channels(dev)  # (3,), (3,) for Na, K, leak
    I_ext = tensors.stimulus_currents(params["I_max"], dev)  # (B,) µA/cm²
    phi = tensors.temperature_factor(params["temp"])
    y0 = tensors.initial_state(len(I_ext), dev)  # (4, B) = (V, m, h, n) at rest

    times, traj = solve.solve(y0, I_ext, g, E, tensors.C_M, phi, params["t_end"])  # (T,), (T, 4, B)
    spikes = solve.spike_times(times, traj[:, 0])
    rate = solve.firing_rate(spikes, params["t_end"])  # (B,) Hz

    n_spikes = [len(s) for s in spikes]
    fires = [b for b in range(len(I_ext)) if n_spikes[b] > 0]
    repeats = [b for b in range(len(I_ext)) if rate[b] > 0]
    print(
        f"{len(I_ext)} axons, I = 0 … {params['I_max']:g} µA/cm², T = {params['temp']:g} °C (φ = {phi:.2f}), "
        f"{len(times) - 1} RK4 steps of {times[1]:.3f} ms"
    )
    print(f"resting potential {traj[0, 0, 0]:.1f} mV, highest spike peak {traj[:, 0].max():.1f} mV")
    if fires:
        print(f"threshold for one spike: I ≈ {I_ext[fires[0]]:.2f} µA/cm²")
    if repeats:
        b = repeats[0]
        print(
            f"threshold for repetitive firing: I ≈ {I_ext[b]:.2f} µA/cm², starting at {rate[b]:.1f} Hz "
            "(it jumps from zero: a type II neuron)"
        )
    for b in range(0, len(I_ext), max(1, len(I_ext) // 8)):
        print(f"  I = {I_ext[b]:5.2f} µA/cm²: {n_spikes[b]:3d} spikes, steady rate {rate[b]:5.1f} Hz")
    return plot.animate(times, traj, I_ext, rate, n_spikes, out_dir / "hodgkin_huxley.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/biophysics/hodgkin_huxley/equation.py
r"""Physics → tensor translation.

The membrane is a capacitor charged by an injected current I and discharged through three ion channels:

    C dV/dt = I − ḡ_Na m³h (V − E_Na) − ḡ_K n⁴ (V − E_K) − ḡ_L (V − E_L)

Each gate x ∈ {m, h, n} opens with rate α_x(V) and closes with rate β_x(V):  dx/dt = φ [α_x (1 − x) − β_x x].

For a batch of axons b (one per stimulus current) and the channels k = (Na, K, leak):

    Γ_{kb} = (m_b³ h_b,  n_b⁴,  1)              open fraction of channel k
    I^{ion}_b = ḡ_k Γ_{kb} (V_b − E_k)          k summed
    C \dot V_b = I_b − I^{ion}_b
    \dot x_{gb} = φ [α_g(V_b)(1 − x_{gb}) − β_g(V_b) x_{gb}]     g = (m, h, n), no sum
"""

import torch


def _x_over_1_minus_exp(x: torch.Tensor) -> torch.Tensor:
    """x / (1 − e^{−x}), which tends to 1 at x = 0 (where the formula itself would read 0/0)."""
    small = x.abs() < 1e-4
    safe = torch.where(small, torch.ones_like(x), x)
    return torch.where(small, 1.0 + x / 2, safe / -torch.expm1(-safe))


def rates(V: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Opening rates α_gb and closing rates β_gb in 1/ms for the gates g = (m, h, n).   V: (B,) → (3, B), (3, B)."""
    alpha_m = _x_over_1_minus_exp((V + 40.0) / 10.0)  # = 0.1 (V+40) / (1 − e^{−(V+40)/10})
    beta_m = 4.0 * torch.exp(-(V + 65.0) / 18.0)
    alpha_h = 0.07 * torch.exp(-(V + 65.0) / 20.0)
    beta_h = 1.0 / (1.0 + torch.exp(-(V + 35.0) / 10.0))
    alpha_n = 0.1 * _x_over_1_minus_exp((V + 55.0) / 10.0)  # = 0.01 (V+55) / (1 − e^{−(V+55)/10})
    beta_n = 0.125 * torch.exp(-(V + 65.0) / 80.0)
    return torch.stack([alpha_m, alpha_h, alpha_n]), torch.stack([beta_m, beta_h, beta_n])


def steady_state(V: torch.Tensor) -> torch.Tensor:
    """x_∞ = α / (α + β): where each gate settles if V is held fixed.   (B,) → (3, B)"""
    alpha, beta = rates(V)
    return alpha / (alpha + beta)


def open_fraction(gates: torch.Tensor) -> torch.Tensor:
    """Γ_kb = (m³h, n⁴, 1) for k = Na, K, leak.   gates (3, B) = (m, h, n) → (3, B)"""
    m, h, n = gates
    return torch.stack([m**3 * h, n**4, torch.ones_like(m)])


def ionic_current(V: torch.Tensor, gates: torch.Tensor, g: torch.Tensor, E: torch.Tensor) -> torch.Tensor:
    """I^ion_b = ḡ_k Γ_kb (V_b − E_k),  k summed over the three channels.   (B,)"""
    driving = V[None, :] - E[:, None]  # (3, B): how far each ion is from its equilibrium
    return torch.einsum("k,kb,kb->b", g, open_fraction(gates), driving)


def rhs(y: torch.Tensor, I_ext: torch.Tensor, g, E, C_m: float, phi: float) -> torch.Tensor:
    """dy/dt for y of shape (4, B) = (V, m, h, n)."""
    V, gates = y[0], y[1:]
    dV = (I_ext - ionic_current(V, gates, g, E)) / C_m
    alpha, beta = rates(V)
    dgates = phi * (alpha * (1.0 - gates) - beta * gates)  # (3, B), every gate on its own
    return torch.cat([dV[None], dgates])


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/biophysics/hodgkin_huxley/tensors.py
"""Build the concrete tensors: membrane constants, the batch of stimulus currents and the resting state.

Classic squid giant axon (Hodgkin & Huxley 1952) in the modern sign convention, resting potential −65 mV.
Units: V in mV, t in ms, currents in µA/cm², conductances in mS/cm², capacitance in µF/cm².
"""

import torch

from .equation import steady_state

C_M = 1.0  # membrane capacitance, µF/cm²
CHANNELS = [  # name, maximal conductance ḡ_k (mS/cm²), reversal potential E_k (mV)
    ("Na", 120.0, 50.0),
    ("K", 36.0, -77.0),
    ("leak", 0.3, -54.387),  # E_L chosen so that the resting potential is exactly −65 mV
]
V_REST = -65.0
N_CURRENTS = 41  # stimulus currents from 0 to I_max


def channels(device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """ḡ_k (3,) and E_k (3,) for k = Na, K, leak."""
    g = torch.tensor([gk for _, gk, _ in CHANNELS], device=device)
    E = torch.tensor([Ek for _, _, Ek in CHANNELS], device=device)
    return g, E


def stimulus_currents(I_max: float, device: torch.device) -> torch.Tensor:
    """I_b, shape (B,): one axon per current, evenly spaced from 0 to I_max."""
    return torch.linspace(0.0, I_max, N_CURRENTS, device=device)


def temperature_factor(temp: float) -> float:
    """φ = 3^((T − 6.3 °C)/10): the gates open and close 3× faster for every 10 °C (Q10 = 3)."""
    return 3.0 ** ((temp - 6.3) / 10.0)


def initial_state(n_axons: int, device: torch.device) -> torch.Tensor:
    """y0 of shape (4, B) = (V, m, h, n): every axon at rest, gates at their resting steady state."""
    V = torch.full((n_axons,), V_REST, device=device)
    return torch.cat([V[None], steady_state(V)])


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/biophysics/hodgkin_huxley/solve.py
"""Integrate the Hodgkin–Huxley equations with RK4, then count spikes and measure the firing rate."""

import torch

from problems._common.integrators import integrate

from .equation import rhs

T_ON = 10.0  # ms: the current step switches on here (the axon rests before)


def time_step(phi: float) -> float:
    """0.05 ms resolves the sub-millisecond Na gate m with RK4; warmer axons (larger φ) need proportionally less."""
    return 0.05 / max(1.0, phi)


def solve(y0: torch.Tensor, I_ext: torch.Tensor, g, E, C_m: float, phi: float, t_end: float):
    """Return (times (T,), trajectory (T, 4, B)): a current step I_b from t = T_ON to t_end, all axons at once."""
    dt = time_step(phi)
    n = int(round(t_end / dt))
    zero = torch.zeros_like(I_ext)
    traj = integrate(lambda y, t: rhs(y, I_ext if t >= T_ON else zero, g, E, C_m, phi), y0, dt, n)
    times = torch.arange(n + 1, device=y0.device) * dt
    return times, traj


def spike_times(times: torch.Tensor, V: torch.Tensor) -> list[torch.Tensor]:
    """Upward crossings of 0 mV, linearly interpolated: one tensor of spike times (ms) per axon b.   V: (T, B)"""
    up = (V[:-1] < 0) & (V[1:] >= 0)  # (T−1, B)
    out = []
    for b in range(V.shape[1]):
        i = up[:, b].nonzero().squeeze(1)
        v0, v1 = V[i, b], V[i + 1, b]
        out.append(times[i] + (0 - v0) / (v1 - v0) * (times[i + 1] - times[i]))
    return out


def firing_rate(spikes: list[torch.Tensor], t_end: float, settle: float = 40.0) -> torch.Tensor:
    """Steady firing rate in Hz from the mean interspike interval after the onset transient; 0 if it stops firing.

    Needs at least two spikes after T_ON + settle; returns a (B,) tensor.
    """
    rate = []
    for s in spikes:
        late = s[s > T_ON + settle]
        rate.append(1000.0 * (len(late) - 1) / (late[-1] - late[0]).item() if len(late) >= 2 else 0.0)
    return torch.tensor(rate)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/biophysics/hodgkin_huxley/plot.py
"""Animate Hodgkin–Huxley: voltage traces for four stimulus currents, the gates m, h, n, and the f–I curve."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from .solve import T_ON

SHOWN = [0.1, 0.25, 0.5, 1.0]  # traces drawn for these fractions of I_max


def animate(
    times: torch.Tensor,
    traj: torch.Tensor,
    I_ext: torch.Tensor,
    rate: torch.Tensor,
    n_spikes: list[int],
    path: Path,
) -> Path:
    stride = max(1, len(times) // 1500)
    t = times[::stride].cpu().numpy()
    y = traj[::stride].cpu().numpy()  # (T, 4, B)
    cur, f = I_ext.cpu().numpy(), rate.cpu().numpy()
    shown = [int(np.abs(cur - s * cur[-1]).argmin()) for s in SHOWN]
    colors = [f"C{k}" for k in range(len(shown))]

    fig = plt.figure(figsize=(8.4, 4.8))
    outer = fig.add_gridspec(1, 2, width_ratios=[1.3, 1], wspace=0.25)
    grid = outer[0].subgridspec(len(shown), 1, hspace=0.12)
    right = outer[1].subgridspec(2, 1, hspace=0.55)

    # --- membrane potential of four axons ---
    traces = []
    for k, b in enumerate(shown):
        ax = fig.add_subplot(grid[k])
        ax.set_xlim(0, t[-1])
        ax.set_ylim(-85, 90)  # headroom above the spikes for the label
        ax.set_yticks([-65, 0, 40])
        ax.axvspan(T_ON, t[-1], color="#fef3c7", lw=0)  # the current is on
        ax.text(
            0.99,
            0.96,
            f"I = {cur[b]:.1f} µA/cm²",
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=8,
            color=colors[k],
        )
        traces.append(ax.plot([], [], lw=1.3, color=colors[k])[0])
        if k < len(shown) - 1:
            ax.tick_params(labelbottom=False)
        if k == 0:
            ax.set_title("membrane potential V (mV); shaded: current on", fontsize=10)
    ax.set_xlabel("time (ms)")

    # --- f–I curve: steady firing rate against current ---
    ax_f = fig.add_subplot(right[0])
    single = np.array([n > 0 and r == 0 for n, r in zip(n_spikes, f)])
    rep = f > 0
    ax_f.plot(cur[rep], f[rep], "o-", ms=3, lw=1, color="#374151", label="repetitive firing")
    ax_f.plot(cur[single], np.zeros(single.sum()), "x", ms=5, color="#dc2626", label="single spike only")
    ax_f.plot(
        cur[~rep & ~single], np.zeros((~rep & ~single).sum()), "o", ms=3, mfc="white", color="#9ca3af", label="no spike"
    )
    for k, b in enumerate(shown):
        ax_f.plot([cur[b]], [f[b]], "o", ms=8, mfc="none", mew=1.5, color=colors[k])
    ax_f.set_xlim(-0.03 * cur[-1], cur[-1] * 1.03)
    ax_f.set_ylim(-5, max(f.max(), 1) * 1.15)
    ax_f.set_xlabel("stimulus I (µA/cm²)", labelpad=1)
    ax_f.set_ylabel("rate (Hz)")
    ax_f.set_title("f–I curve", fontsize=10)
    ax_f.legend(loc="upper left", fontsize=7)

    # --- gating variables of the strongest shown stimulus ---
    ax_g = fig.add_subplot(right[1])
    ax_g.set_xlim(T_ON - 5, min(t[-1], T_ON + 45))
    ax_g.set_ylim(0, 1)
    ax_g.set_xlabel("time (ms)", labelpad=1)
    ax_g.set_title(f"gates at I = {cur[shown[-1]]:.1f} µA/cm²", fontsize=10)
    gate_colors = ["#2563eb", "#f59e0b", "#16a34a"]
    gates = [ax_g.plot([], [], lw=1.5, color=c)[0] for c in gate_colors]
    for j, name in enumerate("mhn"):  # label each gate at its resting value, left of the step
        ax_g.text(T_ON - 4, y[0, 1 + j, shown[-1]] + 0.04, name, color=gate_colors[j], fontsize=10, style="italic")

    def update(step: int):
        for k, b in enumerate(shown):
            traces[k].set_data(t[: step + 1], y[: step + 1, 0, b])
        for j in range(3):
            gates[j].set_data(t[: step + 1], y[: step + 1, 1 + j, shown[-1]])

    fig.subplots_adjust(left=0.07, right=0.98, top=0.93, bottom=0.1)
    return save_gif(fig, update, frame_indices(len(t), 90), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'I_max': 20.0,
    't_end': 150.0,
    'temp': 6.3
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.biophysics.hodgkin_huxley")]:
    del sys.modules[name]
main = importlib.import_module("problems.biophysics.hodgkin_huxley.main")
problem_dir = Path("problems/biophysics/hodgkin_huxley")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
